In [1]:
#Each day, the energy supply and demand vary due to weather and usage patterns. You want to know
#what is the probability that the system meets demand every day for 30 days?

In [6]:
#Step 1 – Model the Uncertainty
#Define daily variation using normal distributions
#Solar output: mean = 45 kWh, standard deviation = 5
#Energy demand: mean = 40 kWh, standard deviation = 6

# Simulation settings
supply_mean <- 45
supply_sd <- 5
demand_mean <- 40
demand_sd <- 6

In [10]:
#Step 2 – Simulate 1,000 Months
#Each simulation represents a 30-day period. 
#We generate supply and demand for each day and check if supply met demand every day.
results <- replicate(1000, {
  supply <- rnorm(30, mean = supply_mean, sd = supply_sd)
  demand <- rnorm(30, mean = demand_mean, sd = demand_sd)
  all(supply >= demand)
})
#The all() function checks if supply was sufficient on all 30 days.

ERROR: Error: object 'supply' not found


In [11]:
#The individual check if the supply was sufficient on all 30 days
 supply <- rnorm(30, mean = supply_mean, sd = supply_sd)
  demand <- rnorm(30, mean = demand_mean, sd = demand_sd)

supply >= demand        # See TRUE/FALSE for each of the 30 days
all(supply >= demand)   # Did ALL 30 days pass?

#all() is strict, is there's one FAIL the whole MONTH fails.

[1]  TRUE  TRUE  TRUE  TRUE  TRUE FALSE  TRUE FALSE  TRUE  TRUE FALSE  TRUE
[13]  TRUE  TRUE  TRUE  TRUE  TRUE  TRUE FALSE FALSE  TRUE  TRUE FALSE FALSE
[25] FALSE FALSE FALSE  TRUE  TRUE  TRUE

[1] FALSE

In [4]:
#Step 3 – Estimate the Probability
#Calculate how often the system succeeded
mean(results)

# the solar grid almost never survives a full 30 days without a shortfall.

[1] 0

In [16]:
#Step 4 – Try Adjusting the Inputs

    #Increase demand_sd to simulate volatile usage
    #Reduce supply_mean to simulate cloudy conditions
    #Add battery buffer logic to store excess energy

#Simulation settings
supply_mean <- 25
supply_sd <- 5
demand_mean <- 40
demand_sd <- 10
battery_max  <- 20   # maximum battery storage (kWh)
battery_start <- 10  # starting charge (kWh)

In [18]:
#Simulating 30-day month with battery
simulate_month <- function() {
  supply  <- rnorm(30, mean = supply_mean, sd = supply_sd)
  demand  <- rnorm(30, mean = demand_mean, sd = demand_sd)
  
  battery <- battery_start  # battery level at start

     for (day in 1:30) {
    excess <- supply[day] - demand[day]  # positive = surplus, negative = shortfall
    
    battery <- battery + excess          # charge or drain battery
    battery <- max(0, battery)           # battery can't go below 0
    battery <- min(battery_max, battery) # battery can't exceed max capacity

          # If battery hit 0, demand was not met
    if (battery == 0 && excess < 0) {
      return(FALSE)  # system failed today
    }
  }
  return(TRUE)  # survived all 30 days
}

In [19]:
#Running simulations
results <- replicate(1000, simulate_month())
mean(results)

[1] 0